# Run inference a pre-trained model

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

import orbax
from orbax import checkpoint
from jax.sharding import SingleDeviceSharding

from src.model.model import NanoLLM
from src.inference.generate import generate_text
from src.config import (
    RECOMMENDED_NEW_TOKENS,
    RECOMMENDED_TEMPERATURE,
    InferenceConfig,
    ModelConfig,
    TokenizerConfig,
)
from src.paths import CHECKPOINTS_DIR

## [Manual] Load pre-trained model from checkpoints
This section uses the manual approach established in the previous notebook.
A later section demonstrates a more consolidated approach

In [ ]:
# import model and default configuration
model_config = ModelConfig()
model = NanoLLM(model_config)

In [ ]:
from src.training.checkpoint import get_latest_checkpoint

cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

checkpoint_source = get_latest_checkpoint()
if checkpoint_source is None:
    raise FileNotFoundError(f"No checkpoint bundles found in {CHECKPOINTS_DIR}")
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [ ]:
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

restored_state = checkpointer.restore(
    checkpoint_source / "weights.orbax",
    item=nnx.state(model),
    restore_args=restore_args)

In [ ]:
# Update the model with the loaded checkpoints
nnx.update(model,restored_state)

In [ ]:
# Manually construct tokenizer config (required for inference)
tokenizer_config = TokenizerConfig(delimiter="<|endoftext|>", name="gpt2")
tokenizer = tokenizer_config.tokenizer

# NB: tokenizer and delimiter are stored together in TokenizerConfig because they are both essential properties that describe how a model was trained.
# This pair should not be coupled directly to the model itself because they might change if we retrain the model.

## [Consolidated] Load pre-trained model from checkpoints 
This section uses a newly developed endpoint that consolidates the steps required for to reconstruct a model from scratch using checkpoints that include training metadata.

_NB: This endpoint requires that `metadata.json` is complete within the checkpoint directory (bespoke to this app). If you only have access to the orbax weights, then use the manual approach in the previous section._ 

In [ ]:
from src.training.checkpoint import get_latest_checkpoint, restore_from_checkpoint

checkpoint_source = get_latest_checkpoint()

# 3-tuple unpack: model + tokenizer_config + metadata in a single read
model2, tokenizer_config2, metadata2 = restore_from_checkpoint(checkpoint_source)
tokenizer2 = tokenizer_config2.tokenizer


print(f"\nLoaded model config: \n\t{model2.config}")
print(f"\nLoaded tokenizer config: \n\t{tokenizer_config2}")
print(f"\nTotal epochs trained: \n\t{metadata2.cumulative_epochs_completed}")

## Run inference on the pre-trained model

### Option A: Experimental, hard-coded method

In [ ]:
# Define method for creating a story
def create_story(story_prompt, max_new_tokens, temperature):

    # Convert the text prompt to token IDs using the tokenizer
    start_tokens = tokenizer2.encode(story_prompt)

    # Ensure max_new_tokens is an integer
    # Underlying inference functions require that max_new_tokens is an integer, 
    # but some callers (e.g. Gradio sliders) return floats for this value
    max_new_tokens = int(max_new_tokens)
    
    inference_config = InferenceConfig(max_new_tokens=max_new_tokens, temperature=temperature)

    # Extracted to module
    generated_text = generate_text(
        model=model2, 
        tokenizer_config=tokenizer_config2, 
        inference_config=inference_config, 
        start_tokens=start_tokens
    )

    return generated_text

In [ ]:
# Call the method to exercise inference
# Passing the InferenceConfig defaults aligns this notebook with the CLI and the web demo, which were developed later.
# FYI: The result here will likely be unsatisfactory if the model was trained for a short time on a small training set.
create_story(
    "I was eating a bowl of raspberries when I noticed that",
    InferenceConfig.max_new_tokens,
    InferenceConfig.temperature,
)

### Option B. CLI script 

_NB: Cells using `!` shell magic prints a `RuntimeWarning` about `os.fork()`. JAX raises it on any fork because JAX is multithreaded and forking a multithreaded
process can deadlock. It is harmless in the cases on this notebook because the forked child immediately replaces itself with the `uv run` command, discarding the inherited JAX state before it can deadlock._

In [ ]:
# shell magic allows us to run the CLI script directly from the notebook

# Enter the input phrase as part of the CLI command (`--prompt` required). Falls back to the default temperature and max tokens defined in `InferenceConfig`.
# If checkpoint bundle is not specified in a separate optional flag, the command builds the model using the most recent checkpoint.
!uv run nanollm-generate --prompt "I was eating a bowl of raspberries when I noticed that"

In [ ]:
# The CLI script can accept optional flags for research purposes. 
# For example, adding a seed value results in a response that remains stable across multiple calls. 
!uv run nanollm-generate --prompt "I wore a hat to the" --seed 33

In [ ]:
# Run it again with the same seed to receive the same response (assuming all other variables are held constant)
!uv run nanollm-generate --prompt "I wore a hat to the" --seed 33

### Option C. Web UI

#### original experimental code

In [ ]:
import gradio as gr

# define demo
demo = gr.Interface(

    # the inference method developed previously
    fn=create_story,

    inputs=[
        gr.Textbox(label="Starter Prompt"),   
        # Bounds and defaults come from config so this demo matches the CLI and `nanollm-demo`.
        gr.Slider(
            minimum=RECOMMENDED_NEW_TOKENS.minimum,
            maximum=RECOMMENDED_NEW_TOKENS.maximum,
            value=InferenceConfig.max_new_tokens,
            step=1,
            label="Max Tokens",
        ),
        # Temperature minimum is > 0 because sampling divides logits by temperature.
        gr.Slider(
            minimum=RECOMMENDED_TEMPERATURE.minimum,
            maximum=RECOMMENDED_TEMPERATURE.maximum,
            value=InferenceConfig.temperature,
            step=0.01,
            label="Temperature",
        ),
    ],
    
    outputs=["text"]
)

# launch demo; share=True will provide the local url
demo.launch(share=True)

#### launch via productionalized script
This hooks into the extracted inference code in `src/`.

In [ ]:
# shell magic allows us to run the script directly from the notebook
# NB: This results in the same `os.fork()` warning (expected and harmless in this case) described under Option B.
!uv run nanollm-demo